# Playlist from image

Takes an image, uses a VLM to describe its mood and atmosphere, then retrieves matching tracks from the FMA dataset using CLAP embeddings.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch
from PIL import Image

if "" not in sys.path:
    sys.path.insert(0, "")

import utils
import data_pipeline
import playlist_utils
import image_gen

In [ ]:
IMAGE_PATH = "ny.jpg"

# Qwen2-VL-2B fits in ~6 GB VRAM (fp16). Larger option: "Qwen/Qwen2-VL-7B-Instruct" (~14 GB)
VLM_MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

VLM_PROMPT = (
    "Describe this image and its emotions in rich detail. "
    "Focus on the mood, atmosphere, feelings, colours, energy level, and any narrative "
    "the scene evokes. Imagine you are curating a music playlist inspired by this image — "
    "what does the image sound like? Be expressive and specific."
)

K = 20

SONGS_DIR = utils.DEFAULT_SONGS_DIR
EMB_PATH = utils.DEFAULT_EMB_PATH

CHECKPOINT_URL = utils.DEFAULT_CHECKPOINT_URL
CHECKPOINT_PATH = utils.DEFAULT_CHECKPOINT_PATH

USE_LORA = True
LORA_CKPT_DIR = Path("checkpoints/clap_lora")

DOWNLOAD_FMA = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

PROMPT_SCRAMBLE = True
SCRAMBLE_STD = 0.05

DUAL_KNN = False

GENERATE_IMAGE = True
SAVE_PLAYLIST = True

In [ ]:
audio_paths, audio_embs = data_pipeline.ensure_embeddings(
    emb_path=EMB_PATH,
    songs_dir=SONGS_DIR,
    device=DEVICE,
    download_fma=DOWNLOAD_FMA,
)

Audio tracks on disk: 8000 / 8000
All tracks present — skipping audio download.
Metadata already present — skipping metadata download.
Found 8000 audio file(s) under data/fma_small
Loaded 7997 embeddings from data/embeddings.npz  shape=(7997, 512)
Already embedded : 7997
New to embed     : 3
Checkpoint already present: checkpoints/music_audioset_epoch_15_esc_90.14.pt


/home/nathan/Documents/PlaylistGenerator/.venv/lib/python3.14/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Load the specified checkpoint checkpoints/music_audioset_epoch_15_esc_90.14.pt from users.
Load Checkpoint...
logit_scale_a 	 Loaded
logit_scale_t 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_real.weight 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_imag.weight 	 Loaded
audio_branch.logmel_extractor.melW 	 Loaded
audio_branch.bn0.weight 	 Loaded
audio_branch.bn0.bias 	 Loaded
audio_branch.patch_embed.proj.weight 	 Loaded
audio_branch.patch_embed.proj.bias 	 Loaded
audio_branch.patch_embed.norm.weight 	 Loaded
audio_branch.patch_embed.norm.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm1.weight 	 Loaded
audio_branch.layers.0.blocks.0.norm1.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.relative_position_bias_table 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.bias 	 Loaded
audio_branch.layers.0.bl

/home/nathan/Documents/PlaylistGenerator/.venv/lib/python3.14/site-packages/laion_clap/hook.py:138: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_waveform, _ = librosa.load(f, sr=48000)
[src/libmpg123/parse.c:do_readahead():1083] warning: Cannot read next header, a one-frame stream? Duh...
/home/nathan/Documents/PlaylistGenerator/.venv/lib/python3.14/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/home/nathan/Documents/PlaylistGenerator/.venv/lib/python3.14/site-packages/laion_clap/hook.py:138: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_waveform, _ = librosa.load(f, sr=48000)
/home/nathan/Documents/PlaylistGenerator/.venv/lib/python3.14/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of libr

In [ ]:
if USE_LORA:
    model = utils.load_clap_model_with_lora(
        LORA_CKPT_DIR,
        checkpoint_path=CHECKPOINT_PATH,
        checkpoint_url=CHECKPOINT_URL,
        device=DEVICE,
    )
else:
    model = utils.load_clap_model(CHECKPOINT_PATH, CHECKPOINT_URL, device=DEVICE)

Checkpoint already present: checkpoints/music_audioset_epoch_15_esc_90.14.pt


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Load the specified checkpoint checkpoints/music_audioset_epoch_15_esc_90.14.pt from users.
Load Checkpoint...
logit_scale_a 	 Loaded
logit_scale_t 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_real.weight 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_imag.weight 	 Loaded
audio_branch.logmel_extractor.melW 	 Loaded
audio_branch.bn0.weight 	 Loaded
audio_branch.bn0.bias 	 Loaded
audio_branch.patch_embed.proj.weight 	 Loaded
audio_branch.patch_embed.proj.bias 	 Loaded
audio_branch.patch_embed.norm.weight 	 Loaded
audio_branch.patch_embed.norm.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm1.weight 	 Loaded
audio_branch.layers.0.blocks.0.norm1.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.relative_position_bias_table 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.bias 	 Loaded
audio_branch.layers.0.bl

In [ ]:
from IPython.display import display as _disp
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

image = Image.open(IMAGE_PATH).convert("RGB")
_disp(image)
print(f"Image: {IMAGE_PATH}  ({image.size[0]}×{image.size[1]})")

print(f"\nLoading VLM: {VLM_MODEL_ID} (first run may download weights) …")
_vlm_dtype = torch.float16 if DEVICE == "cuda" else torch.float32

vlm_processor = AutoProcessor.from_pretrained(VLM_MODEL_ID)
vlm = Qwen2VLForConditionalGeneration.from_pretrained(
    VLM_MODEL_ID,
    torch_dtype=_vlm_dtype,
    low_cpu_mem_usage=True,
).to(DEVICE)
vlm.eval()
print("VLM ready.")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": VLM_PROMPT},
        ],
    }
]

prompt_text = vlm_processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)

inputs = vlm_processor(
    text=[prompt_text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
).to(DEVICE)

with torch.no_grad():
    output_ids = vlm.generate(**inputs, max_new_tokens=300, do_sample=False)

generated_ids = [out[len(inp):] for inp, out in zip(inputs.input_ids, output_ids)]
IMAGE_DESCRIPTION = vlm_processor.batch_decode(
    generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0].strip()

print("\nVLM description:\n")
print(IMAGE_DESCRIPTION)

ImportError: cannot import name 'AutoModelForVision2Seq' from 'transformers' (/home/nathan/Documents/PlaylistGenerator/.venv/lib/python3.14/site-packages/transformers/__init__.py)

In [ ]:
text_emb = model.get_text_embedding([IMAGE_DESCRIPTION], use_tensor=False)
text_emb = utils.l2_normalize(np.asarray(text_emb, dtype=np.float32))[0]

if PROMPT_SCRAMBLE:
    text_emb = playlist_utils.scramble_embedding(text_emb, std=SCRAMBLE_STD)
    print(f"Prompt Scramble applied (std={SCRAMBLE_STD})")

print(f"Text embedding shape: {text_emb.shape}")

if DUAL_KNN:
    indices, scores = playlist_utils.dual_anchor_knn(text_emb, audio_embs, k=K)
    score_label = "blended"
else:
    indices, scores = utils.knn_search(text_emb, audio_embs, k=K)
    score_label = "cosine"

print(f"\nPlaylist for: {IMAGE_PATH!r}")
print(f"{'Rank':<5} {score_label.capitalize():>8}  Track")
print("-" * 70)
for rank, (idx, score) in enumerate(zip(indices, scores), start=1):
    track_name = Path(audio_paths[idx]).name
    print(f"{rank:<5} {score:>8.4f}  {track_name}")

In [ ]:
from IPython.display import display, Audio, HTML

TOP_PREVIEW = 5

display(HTML(f"<h3>Top {TOP_PREVIEW} preview — <em>{Path(IMAGE_PATH).name}</em></h3>"))

for rank, (idx, score) in enumerate(zip(indices[:TOP_PREVIEW], scores[:TOP_PREVIEW]), start=1):
    path = audio_paths[idx]
    name = Path(path).name
    display(HTML(
        f"<b>#{rank} &nbsp; {name} &nbsp; "
        f"<span style='color:gray'>score: {score:.4f}</span></b>"
    ))
    display(Audio(path, autoplay=False))

In [ ]:
cover = None
if GENERATE_IMAGE:
    from IPython.display import display as _display

    print("Generating playlist cover image (first run downloads the model ~5 GB) …")
    cover = image_gen.generate_playlist_image(IMAGE_DESCRIPTION, device=DEVICE)
    _display(cover)

In [ ]:
if SAVE_PLAYLIST:
    playlist_dir = playlist_utils.save_playlist(
        Path(IMAGE_PATH).stem,
        track_paths=[audio_paths[i] for i in indices],
        cover_image=cover,
    )
    print(f"→ {playlist_dir.resolve()}")